# 🔬 Notebook 5: Ingeniería de Features Avanzada

## TFM: Predicción de Estrategias de Carrera en Fórmula 1 mediante ML

**Autor:** Francisco José Moreno Bayona  
**Universidad:** UNIR  

---

### Contexto

En el Notebook 4, la optimización de hiperparámetros produjo mejoras marginales.
El análisis de feature importance reveló que **DegradationIndicator** domina con 70-80%
de la importancia, lo que indica que el cuello de botella está en la **información disponible**,
no en la configuración de los modelos.

### Objetivo

Enriquecer el dataset con **nuevas features** derivadas de FastF1 que capturen
aspectos de la estrategia que actualmente no estamos modelando:

| Grupo | Features nuevas | Justificación estratégica |
|-------|----------------|--------------------------|
| **Gaps** | Gap al coche delante/detrás | Decisiones de undercut/overcut |
| **Track Status** | Safety Car, VSC, banderas | Paradas oportunistas bajo SC |
| **Historial de paradas** | Nº paradas ya realizadas | Estrategia 1-stop vs 2-stop |
| **Degradación avanzada** | Tendencia, aceleración | Ritmo de empeoramiento |
| **Circuito** | Duración pit lane, nº vueltas | Coste de parar varía por circuito |
| **Rendimiento relativo** | Delta vs media del campo | Posición competitiva real |

### Metodología
1. Cargar datos crudos del Notebook 1 (pre-normalización)
2. Construir nuevas features
3. Reconstruir datasets de clasificación y regresión
4. Reentrenar modelos con hiperparámetros óptimos del Notebook 4
5. Comparar con resultados anteriores

## 1. Imports y carga de datos base

In [2]:
!python --version

Python 3.9.0


In [3]:
!pip list | findstr "scikit-learn xgboost lightgbm shap pandas numpy fastf1 matplotlib"

matplotlib          3.9.4
matplotlib-inline   0.2.1
numpy               2.0.2
scikit-learn        1.6.1


You should consider upgrading via the 'c:\users\franm\appdata\local\programs\python\python39\python.exe -m pip install --upgrade pip' command.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time, pickle, json, os
from pathlib import Path

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    classification_report, confusion_matrix,
    mean_squared_error, mean_absolute_error, r2_score
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
RANDOM_STATE = 42

print('✅ Imports completados')

✅ Imports completados


In [14]:
# ============================================================================
# CARGAR DATASET CRUDO (pre-normalización, del Notebook 1)
# ============================================================================

df = pd.read_csv('./datasets/dataset_clasificacion.csv')

print(f'Dataset base: {df.shape}')
print(f'Columnas: {list(df.columns)}')
print(f'Temporadas: {sorted(df["Season"].unique())}')
print(f'Carreras: {df.groupby(["Season","Round"]).ngroups}')
print(f'Pilotos: {df["Driver"].nunique()}')

Dataset base: (92991, 18)
Columnas: ['Season', 'Round', 'EventName', 'EventDate', 'Driver', 'LapNumber', 'LapProgress', 'Position', 'TireAge', 'StintNumber', 'IsHard', 'IsMedium', 'IsSoft', 'LapTime_seconds', 'AvgTime_Last3', 'DegradationIndicator', 'target_parada', 'StopLapNumber']
Temporadas: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Carreras: 90
Pilotos: 31


---
## 2. Nuevas features: Gaps entre coches

### ¿Por qué son importantes?

La estrategia de **undercut** consiste en parar ANTES que el rival para ganar
posición gracias a neumáticos frescos. La de **overcut** es lo contrario:
quedarse fuera esperando que el rival pare primero.

Ambas decisiones dependen del **gap** (diferencia de tiempo) con los coches cercanos.
Un gap pequeño (1-2s) incentiva el undercut; un gap grande (5s+) permite esperar.

### Features construidas
- `GapAhead`: Diferencia de tiempo acumulado con el coche de delante (segundos)
- `GapBehind`: Diferencia con el coche de detrás (segundos)
- `IsInDRSWindow`: ¿Está a menos de 1 segundo del coche de delante? (0/1)

In [16]:
# ============================================================================
# FEATURE: GAPS ENTRE COCHES
# ============================================================================

def calcular_gaps(df_carrera):
    '''Calcula gap al coche de delante y detrás para cada vuelta.'''
    df_c = df_carrera.copy()
    
    # Necesitamos Time (acumulado) y Position por vuelta
    gaps_ahead = []
    gaps_behind = []
    
    for lap_num in df_c['LapNumber'].unique():
        lap_data = df_c[df_c['LapNumber'] == lap_num].copy()
        lap_data = lap_data.sort_values('Position')
        
        for i, (idx, row) in enumerate(lap_data.iterrows()):
            pos = row['Position']
            lt = row['LapTime_seconds']
            
            # Gap al coche de delante
            ahead = lap_data[lap_data['Position'] == pos - 1]
            if len(ahead) > 0 and pd.notna(lt):
                gap_a = lt - ahead.iloc[0]['LapTime_seconds']
                gaps_ahead.append((idx, gap_a))
            else:
                gaps_ahead.append((idx, np.nan))
            
            # Gap al coche de detrás
            behind = lap_data[lap_data['Position'] == pos + 1]
            if len(behind) > 0 and pd.notna(lt):
                gap_b = behind.iloc[0]['LapTime_seconds'] - lt
                gaps_behind.append((idx, gap_b))
            else:
                gaps_behind.append((idx, np.nan))
    
    # Asignar
    df_c['GapAhead'] = np.nan
    df_c['GapBehind'] = np.nan
    for idx, val in gaps_ahead:
        df_c.loc[idx, 'GapAhead'] = val
    for idx, val in gaps_behind:
        df_c.loc[idx, 'GapBehind'] = val
    
    return df_c

# Aplicar por carrera
print('Calculando gaps entre coches...')
df = df.groupby(['Season', 'Round'], group_keys=False).apply(calcular_gaps)

# Ventana DRS (< 1 segundo del coche de delante)
df['IsInDRSWindow'] = (df['GapAhead'].abs() < 1.0).astype(int)

# Rellenar NaN con mediana (P1 no tiene gap delante, P20 no tiene detrás)
df['GapAhead'] = df['GapAhead'].fillna(df['GapAhead'].median())
df['GapBehind'] = df['GapBehind'].fillna(df['GapBehind'].median())

print(f'✅ Gaps calculados')
print(f'  GapAhead: media={df["GapAhead"].mean():.2f}s, mediana={df["GapAhead"].median():.2f}s')
print(f'  GapBehind: media={df["GapBehind"].mean():.2f}s')
print(f'  En ventana DRS: {df["IsInDRSWindow"].mean():.1%}')

Calculando gaps entre coches...


KeyError: 'Season'

## 3. Nuevas features: Historial de paradas y stint

### ¿Por qué son importantes?

Un piloto que ya ha hecho 1 parada tiene menor probabilidad de parar de nuevo
(si la carrera es de 1 parada) o mayor si es de 2-3 paradas.
El número de stint actual y las paradas ya realizadas son información clave
que los ingenieros de estrategia usan constantemente.

### Features construidas
- `NumStopsDone`: Número de paradas ya realizadas en la carrera
- `LapsInCurrentStint`: Vueltas completadas en el stint actual
- `RemainingLaps`: Vueltas restantes hasta el final de la carrera
- `RemainingLapsPct`: Porcentaje de carrera restante

In [11]:
# ============================================================================
# CARGAR DATASET CRUDO (del Notebook 1)
# ============================================================================

df = pd.read_csv('./datasets/dataset_clasificacion.csv')

print(f'Dataset base: {df.shape}')
print(f'\nColumnas disponibles:')
for c in df.columns:
    print(f'  - {c}')

# Comprobar si existe una columna de tiempo acumulado de carrera
columnas_tiempo = [c for c in df.columns if c.lower() in
                   ['time', 'racetime', 'cumulativetime', 'lapstarttime']]
print(f'\nColumnas de tiempo acumulado detectadas: {columnas_tiempo if columnas_tiempo else "NINGUNA"}')

Dataset base: (92991, 18)

Columnas disponibles:
  - Season
  - Round
  - EventName
  - EventDate
  - Driver
  - LapNumber
  - LapProgress
  - Position
  - TireAge
  - StintNumber
  - IsHard
  - IsMedium
  - IsSoft
  - LapTime_seconds
  - AvgTime_Last3
  - DegradationIndicator
  - target_parada
  - StopLapNumber

Columnas de tiempo acumulado detectadas: NINGUNA


In [12]:
# ============================================================================
# FEATURES: HISTORIAL DE PARADAS Y CARRERA RESTANTE
# ============================================================================

def calcular_historial_paradas(group):
    '''Calcula paradas acumuladas y vueltas restantes por piloto.'''
    g = group.sort_values('LapNumber').copy()
    
    # Número de paradas YA realizadas (acumulado ANTES de esta vuelta)
    g['NumStopsDone'] = g['target_parada'].shift(1, fill_value=0).cumsum()
    
    # Vueltas en el stint actual (reinicia con cada parada)
    stint_id = g['NumStopsDone'].diff().fillna(0).ne(0).cumsum()
    g['LapsInCurrentStint'] = g.groupby(stint_id).cumcount() + 1
    
    # Vueltas restantes
    if 'TotalLaps' in g.columns:
        total = g['TotalLaps'].iloc[0]
    else:
        total = g['LapNumber'].max()
    
    g['RemainingLaps'] = total - g['LapNumber']
    g['RemainingLapsPct'] = g['RemainingLaps'] / total
    
    return g

print('Calculando historial de paradas y carrera restante...')
df = df.groupby(['Season', 'Round', 'Driver'], group_keys=False).apply(calcular_historial_paradas)

print(f'✅ Historial calculado')
print(f'  NumStopsDone: {df["NumStopsDone"].value_counts().sort_index().to_dict()}')
print(f'  RemainingLaps: media={df["RemainingLaps"].mean():.1f}')

Calculando historial de paradas y carrera restante...
✅ Historial calculado
  NumStopsDone: {0: 31795, 1: 38534, 2: 18204, 3: 3736, 4: 669, 5: 51, 6: 2}
  RemainingLaps: media=28.7


## 4. Nuevas features: Degradación avanzada

### ¿Por qué?

En el Notebook 3, `DegradationIndicator` (diferencia del tiempo actual vs. media de 3 vueltas)
fue la feature más importante. Pero es una medida instantánea.
Podemos capturar mejor la dinámica de degradación con:

- **Tendencia**: ¿Los tiempos empeoran o mejoran? (pendiente lineal últimas 5 vueltas)
- **Aceleración**: ¿La degradación se acelera? (derivada segunda)
- **Delta vs mejor tiempo**: Diferencia con el mejor tiempo propio en la carrera
- **Delta vs media del campo**: ¿Va más rápido o lento que la media en esa vuelta?

In [ ]:
# ============================================================================
# FEATURES: DEGRADACIÓN AVANZADA
# ============================================================================

def calcular_degradacion_avanzada(group):
    '''Calcula métricas avanzadas de degradación del neumático.'''
    g = group.sort_values('LapNumber').copy()
    lt = g['LapTime_seconds']
    
    # Tendencia: pendiente de tiempos en últimas 5 vueltas (rolling slope)
    # Positivo = empeorando, Negativo = mejorando
    def rolling_slope(series, window=5):
        slopes = []
        for i in range(len(series)):
            if i < window - 1:
                slopes.append(np.nan)
            else:
                y = series.iloc[i-window+1:i+1].values
                if np.any(np.isnan(y)):
                    slopes.append(np.nan)
                else:
                    x = np.arange(window)
                    slope = np.polyfit(x, y, 1)[0]
                    slopes.append(slope)
        return slopes
    
    g['DegradationTrend'] = rolling_slope(lt, window=5)
    
    # Aceleración: cambio en la tendencia (¿se acelera la degradación?)
    g['DegradationAccel'] = g['DegradationTrend'].diff()
    
    # Delta vs mejor tiempo propio en la carrera
    best_time = lt.expanding().min()
    g['DeltaVsBestLap'] = lt - best_time
    
    # Media de último stint (rendimiento en neumáticos actuales)
    g['AvgTime_CurrentStint'] = g.groupby('StintNumber')['LapTime_seconds'].transform(
        lambda x: x.expanding().mean())
    
    return g

print('Calculando degradación avanzada...')
df = df.groupby(['Season', 'Round', 'Driver'], group_keys=False).apply(calcular_degradacion_avanzada)

# Rellenar NaN iniciales con 0 (primeras vueltas sin ventana suficiente)
for col in ['DegradationTrend', 'DegradationAccel', 'DeltaVsBestLap', 'AvgTime_CurrentStint']:
    df[col] = df[col].fillna(0)

print(f'✅ Degradación avanzada calculada')
print(f'  DegradationTrend: media={df["DegradationTrend"].mean():.4f}')
print(f'  DeltaVsBestLap: media={df["DeltaVsBestLap"].mean():.2f}s')

## 5. Nuevas features: Rendimiento relativo al campo

### ¿Por qué?

El tiempo absoluto de vuelta varía mucho entre circuitos (70s en Austria vs 100s en Singapur).
Lo que importa estratégicamente es cómo rinde un piloto **respecto al campo** en esa vuelta.
Un piloto que pierde 2s/vuelta respecto a la media tiene urgencia de parar;
uno que va 1s más rápido puede estirar el stint.

In [ ]:
# ============================================================================
# FEATURES: RENDIMIENTO RELATIVO AL CAMPO
# ============================================================================

def calcular_rendimiento_relativo(df_carrera):
    '''Delta del piloto vs. media del campo en cada vuelta.'''
    dc = df_carrera.copy()
    
    # Media del campo por vuelta
    lap_means = dc.groupby('LapNumber')['LapTime_seconds'].transform('mean')
    lap_stds = dc.groupby('LapNumber')['LapTime_seconds'].transform('std')
    
    # Delta respecto a la media (positivo = más lento que la media)
    dc['DeltaVsFieldMean'] = dc['LapTime_seconds'] - lap_means
    
    # Z-score respecto al campo (normalizado por desviación)
    dc['ZScoreVsField'] = np.where(lap_stds > 0,
        (dc['LapTime_seconds'] - lap_means) / lap_stds, 0)
    
    # Posición relativa normalizada (0 = líder, 1 = último)
    n_drivers = dc.groupby('LapNumber')['Driver'].transform('count')
    dc['PositionNormalized'] = (dc['Position'] - 1) / (n_drivers - 1).clip(lower=1)
    
    return dc

print('Calculando rendimiento relativo al campo...')
df = df.groupby(['Season', 'Round'], group_keys=False).apply(calcular_rendimiento_relativo)

# Rellenar NaN
for col in ['DeltaVsFieldMean', 'ZScoreVsField', 'PositionNormalized']:
    df[col] = df[col].fillna(0)

print(f'✅ Rendimiento relativo calculado')
print(f'  DeltaVsFieldMean: media={df["DeltaVsFieldMean"].mean():.3f}s')
print(f'  ZScoreVsField: media={df["ZScoreVsField"].mean():.3f}')

## 6. Resumen de features y reconstrucción de datasets

In [ ]:
# ============================================================================
# DEFINIR CONJUNTO COMPLETO DE FEATURES
# ============================================================================

FEATURES_ORIGINAL = [
    'LapNumber', 'LapProgress', 'Position', 'TireAge', 'StintNumber',
    'IsHard', 'IsMedium', 'IsSoft',
    'LapTime_seconds', 'AvgTime_Last3', 'DegradationIndicator',
]

FEATURES_NUEVAS = [
    # Gaps
    'GapAhead', 'GapBehind', 'IsInDRSWindow',
    # Historial de paradas
    'NumStopsDone', 'LapsInCurrentStint', 'RemainingLaps', 'RemainingLapsPct',
    # Degradación avanzada
    'DegradationTrend', 'DegradationAccel', 'DeltaVsBestLap', 'AvgTime_CurrentStint',
    # Rendimiento relativo
    'DeltaVsFieldMean', 'ZScoreVsField', 'PositionNormalized',
]

FEATURES_ALL = FEATURES_ORIGINAL + FEATURES_NUEVAS

# Verificar disponibilidad
features_disponibles = [f for f in FEATURES_ALL if f in df.columns]
features_faltantes = [f for f in FEATURES_ALL if f not in df.columns]

print(f'Features originales: {len(FEATURES_ORIGINAL)}')
print(f'Features nuevas: {len(FEATURES_NUEVAS)}')
print(f'Total disponibles: {len(features_disponibles)}')
if features_faltantes:
    print(f'⚠️ Faltantes: {features_faltantes}')
else:
    print(f'✅ Todas las features presentes')

print(f'\nFeatures nuevas añadidas:')
for f in FEATURES_NUEVAS:
    if f in df.columns:
        print(f'  ✓ {f:25s} → media={df[f].mean():.3f}, nulls={df[f].isnull().sum()}')

In [ ]:
# ============================================================================
# RECONSTRUIR DATASETS CON NUEVAS FEATURES
# ============================================================================

# --- Target regresión: LapsUntilNextStop ---
df_sorted = df.sort_values(['Season', 'Round', 'Driver', 'LapNumber']).copy()

def calcular_laps_until_stop(group):
    g = group.copy()
    stop_laps = g[g['target_parada'] == 1]['LapNumber'].values
    laps_until = []
    for _, row in g.iterrows():
        lap = row['LapNumber']
        future_stops = stop_laps[stop_laps >= lap]
        laps_until.append(future_stops[0] - lap if len(future_stops) > 0 else -1)
    g['LapsUntilNextStop'] = laps_until
    return g

print('Calculando LapsUntilNextStop...')
df_with_target = df_sorted.groupby(['Season', 'Round', 'Driver'], group_keys=False).apply(calcular_laps_until_stop)

# --- Datasets ---
# Clasificación: todas las vueltas
df_clasif = df_with_target.copy()

# Regresión: solo vueltas con parada futura
df_regres = df_with_target[df_with_target['LapsUntilNextStop'] >= 0].copy()

# --- División temporal (80/20) ---
df_clasif = df_clasif.sort_values('EventDate').reset_index(drop=True)
df_regres = df_regres.sort_values('EventDate').reset_index(drop=True)

n_c = int(len(df_clasif) * 0.8)
n_r = int(len(df_regres) * 0.8)

train_c = df_clasif.iloc[:n_c]
test_c = df_clasif.iloc[n_c:]
train_r = df_regres.iloc[:n_r]
test_r = df_regres.iloc[n_r:]

print(f'\nDatasets reconstruidos:')
print(f'  Clasificación: train={len(train_c):,} | test={len(test_c):,}')
print(f'  Regresión: train={len(train_r):,} | test={len(test_r):,}')
print(f'  Features: {len(features_disponibles)} ({len(FEATURES_ORIGINAL)} originales + {len(FEATURES_NUEVAS)} nuevas)')

In [ ]:
# ============================================================================
# PREPARAR X, y
# ============================================================================

X_train_c = train_c[features_disponibles].copy()
y_train_c = train_c['target_parada'].copy()
X_test_c = test_c[features_disponibles].copy()
y_test_c = test_c['target_parada'].copy()

X_train_r = train_r[features_disponibles].copy()
y_train_r = train_r['LapsUntilNextStop'].copy()
X_test_r = test_r[features_disponibles].copy()
y_test_r = test_r['LapsUntilNextStop'].copy()

# Normalizado para MLP
scaler_c = StandardScaler()
scaler_r = StandardScaler()
features_num = [f for f in features_disponibles if f not in ['IsHard', 'IsMedium', 'IsSoft', 'IsInDRSWindow']]

X_train_c_norm = X_train_c.copy()
X_test_c_norm = X_test_c.copy()
X_train_c_norm[features_num] = scaler_c.fit_transform(X_train_c[features_num])
X_test_c_norm[features_num] = scaler_c.transform(X_test_c[features_num])

X_train_r_norm = X_train_r.copy()
X_test_r_norm = X_test_r.copy()
X_train_r_norm[features_num] = scaler_r.fit_transform(X_train_r[features_num])
X_test_r_norm[features_num] = scaler_r.transform(X_test_r[features_num])

# Peso para desbalance
n_neg = (y_train_c == 0).sum()
n_pos = (y_train_c == 1).sum()
scale_pos_weight = n_neg / n_pos

print(f'X_train_c: {X_train_c.shape} | X_test_c: {X_test_c.shape}')
print(f'X_train_r: {X_train_r.shape} | X_test_r: {X_test_r.shape}')
print(f'Scale pos weight: {scale_pos_weight:.1f}')
print(f'\n✅ Datos preparados con {len(features_disponibles)} features')

## 7. Reentrenar modelos con hiperparámetros óptimos del Notebook 4

Usamos exactamente los mismos hiperparámetros del Notebook 4,
pero ahora con **25 features** en lugar de 11.
Si el cuello de botella eran las features (como diagnosticamos),
deberíamos ver mejoras significativas.

In [ ]:
# ============================================================================
# CARGAR HIPERPARÁMETROS ÓPTIMOS DEL NOTEBOOK 4
# ============================================================================

with open('./datasets/prepared/mejores_hiperparametros.json', 'r') as f:
    best_params = json.load(f)

# Reconstruir modelos con hiperparámetros óptimos
modelos_clasif = {
    'Decision Tree': DecisionTreeClassifier(
        class_weight='balanced', random_state=RANDOM_STATE,
        **best_params['clasificacion']['Decision Tree']
    ),
    'Random Forest': RandomForestClassifier(
        class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE,
        **best_params['clasificacion']['Random Forest']
    ),
    'XGBoost': XGBClassifier(
        scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=RANDOM_STATE,
        **best_params['clasificacion']['XGBoost']
    ),
    'LightGBM': LGBMClassifier(
        is_unbalance=True, verbose=-1, random_state=RANDOM_STATE,
        **best_params['clasificacion']['LightGBM']
    ),
    'MLP': MLPClassifier(
        early_stopping=True, validation_fraction=0.1, random_state=RANDOM_STATE,
        **{k: tuple(v) if k == 'hidden_layer_sizes' else v
           for k, v in best_params['clasificacion']['MLP'].items()}
    ),
}

modelos_regres = {
    'Decision Tree': DecisionTreeRegressor(
        random_state=RANDOM_STATE,
        **best_params['regresion']['Decision Tree']
    ),
    'Random Forest': RandomForestRegressor(
        n_jobs=-1, random_state=RANDOM_STATE,
        **best_params['regresion']['Random Forest']
    ),
    'XGBoost': XGBRegressor(
        random_state=RANDOM_STATE,
        **best_params['regresion']['XGBoost']
    ),
    'LightGBM': LGBMRegressor(
        verbose=-1, random_state=RANDOM_STATE,
        **best_params['regresion']['LightGBM']
    ),
    'MLP': MLPRegressor(
        early_stopping=True, validation_fraction=0.1, random_state=RANDOM_STATE,
        **{k: tuple(v) if k == 'hidden_layer_sizes' else v
           for k, v in best_params['regresion']['MLP'].items()}
    ),
}

print('✅ Modelos reconstruidos con hiperparámetros óptimos del Notebook 4')

## 8. Entrenamiento y evaluación — Clasificación

In [ ]:
# ============================================================================
# ENTRENAR Y EVALUAR — CLASIFICACIÓN
# ============================================================================

# Resultados previos (Notebook 4 optimizado)
prev_clasif = {
    'MLP': 0.6958, 'XGBoost': 0.6873, 'Random Forest': 0.6825,
    'LightGBM': 0.6787, 'Decision Tree': 0.6438
}

resultados_clasif_v2 = {}

for nombre, modelo in modelos_clasif.items():
    print(f'\n{"="*60}')
    print(f'Entrenando: {nombre} ({len(features_disponibles)} features)')
    print(f'{"="*60}')
    
    inicio = time.time()
    
    if nombre == 'MLP':
        modelo.fit(X_train_c_norm, y_train_c)
        y_pred = modelo.predict(X_test_c_norm)
        y_proba = modelo.predict_proba(X_test_c_norm)[:, 1]
    else:
        modelo.fit(X_train_c, y_train_c)
        y_pred = modelo.predict(X_test_c)
        y_proba = modelo.predict_proba(X_test_c)[:, 1]
    
    duracion = time.time() - inicio
    f1 = f1_score(y_test_c, y_pred)
    precision = precision_score(y_test_c, y_pred, zero_division=0)
    recall = recall_score(y_test_c, y_pred)
    roc_auc = roc_auc_score(y_test_c, y_proba)
    prev = prev_clasif.get(nombre, 0)
    mejora = f1 - prev
    
    resultados_clasif_v2[nombre] = {
        'F1 (11 feat)': prev,
        'F1 (25 feat)': f1,
        'Mejora': mejora,
        'Precision': precision,
        'Recall': recall,
        'ROC-AUC': roc_auc,
        'Tiempo': duracion
    }
    
    emoji = '📈' if mejora > 0.01 else ('➡️' if mejora > -0.01 else '📉')
    print(f'  {emoji} F1: {prev:.4f} → {f1:.4f} ({mejora:+.4f})')
    print(f'  Precision={precision:.4f}  Recall={recall:.4f}  AUC={roc_auc:.4f}')
    print(f'  Tiempo: {duracion:.1f}s')

print('\n✅ Clasificación completada')

## 9. Entrenamiento y evaluación — Regresión

In [ ]:
# ============================================================================
# ENTRENAR Y EVALUAR — REGRESIÓN
# ============================================================================

prev_regres = {
    'LightGBM': 8.2837, 'Random Forest': 8.2899, 'XGBoost': 8.3635,
    'Decision Tree': 8.5400, 'MLP': 8.6917
}

resultados_regres_v2 = {}

for nombre, modelo in modelos_regres.items():
    print(f'\n{"="*60}')
    print(f'Entrenando: {nombre} ({len(features_disponibles)} features)')
    print(f'{"="*60}')
    
    inicio = time.time()
    
    if nombre == 'MLP':
        modelo.fit(X_train_r_norm, y_train_r)
        y_pred = modelo.predict(X_test_r_norm)
    else:
        modelo.fit(X_train_r, y_train_r)
        y_pred = modelo.predict(X_test_r)
    
    duracion = time.time() - inicio
    rmse = np.sqrt(mean_squared_error(y_test_r, y_pred))
    mae = mean_absolute_error(y_test_r, y_pred)
    r2 = r2_score(y_test_r, y_pred)
    prev = prev_regres.get(nombre, 99)
    mejora = prev - rmse
    
    resultados_regres_v2[nombre] = {
        'RMSE (11 feat)': prev,
        'RMSE (25 feat)': rmse,
        'Mejora RMSE': mejora,
        'MAE': mae,
        'R²': r2,
        'Tiempo': duracion
    }
    
    emoji = '📈' if mejora > 0.1 else ('➡️' if mejora > -0.1 else '📉')
    print(f'  {emoji} RMSE: {prev:.3f} → {rmse:.3f} ({mejora:+.3f})')
    print(f'  MAE={mae:.3f}  R²={r2:.4f}')
    print(f'  Tiempo: {duracion:.1f}s')

print('\n✅ Regresión completada')

## 10. Comparativa: 11 features vs 25 features

In [ ]:
# ============================================================================
# COMPARATIVA VISUAL
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Clasificación ---
df_c = pd.DataFrame(resultados_clasif_v2).T.sort_values('F1 (25 feat)', ascending=False)
x = np.arange(len(df_c))
w = 0.35
axes[0].bar(x - w/2, df_c['F1 (11 feat)'], w, label='11 features (Nb4)', color='#90CAF9')
axes[0].bar(x + w/2, df_c['F1 (25 feat)'], w, label='25 features (Nb5)', color='#1565C0')
for i, m in enumerate(df_c['Mejora']):
    color = '#2E7D32' if m > 0 else '#C62828'
    axes[0].text(i + w/2, df_c['F1 (25 feat)'].iloc[i] + 0.01, f'{m:+.3f}',
                ha='center', fontsize=10, fontweight='bold', color=color)
axes[0].set_xticks(x)
axes[0].set_xticklabels(df_c.index, rotation=20, ha='right')
axes[0].set_ylabel('F1-score')
axes[0].set_title('Clasificación: 11 vs 25 features', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].set_ylim(0, 1)

# --- Regresión ---
df_r = pd.DataFrame(resultados_regres_v2).T.sort_values('RMSE (25 feat)')
x = np.arange(len(df_r))
axes[1].bar(x - w/2, df_r['RMSE (11 feat)'], w, label='11 features (Nb4)', color='#FFAB91')
axes[1].bar(x + w/2, df_r['RMSE (25 feat)'], w, label='25 features (Nb5)', color='#D32F2F')
for i, m in enumerate(df_r['Mejora RMSE']):
    color = '#2E7D32' if m > 0 else '#C62828'
    axes[1].text(i + w/2, df_r['RMSE (25 feat)'].iloc[i] + 0.05, f'{m:+.3f}',
                ha='center', fontsize=10, fontweight='bold', color=color)
axes[1].set_xticks(x)
axes[1].set_xticklabels(df_r.index, rotation=20, ha='right')
axes[1].set_ylabel('RMSE (vueltas)')
axes[1].set_title('Regresión: 11 vs 25 features', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('./datasets/prepared/fig_11_vs_25_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Importancia de features actualizada

¿Las nuevas features aparecen como importantes? ¿Se reduce la dominancia de DegradationIndicator?

In [ ]:
# ============================================================================
# FEATURE IMPORTANCE — CLASIFICACIÓN (25 features)
# ============================================================================

fig, axes = plt.subplots(1, 4, figsize=(24, 7))

for idx, nombre in enumerate(['Decision Tree', 'Random Forest', 'XGBoost', 'LightGBM']):
    modelo = modelos_clasif[nombre]
    importancias = modelo.feature_importances_
    indices = np.argsort(importancias)[::-1][:15]  # Top 15
    
    colors = ['#E53935' if features_disponibles[i] in FEATURES_NUEVAS else '#1565C0'
              for i in indices]
    
    axes[idx].barh(
        [features_disponibles[i] for i in indices],
        importancias[indices],
        color=colors
    )
    axes[idx].set_title(nombre, fontsize=12, fontweight='bold')
    axes[idx].invert_yaxis()

# Leyenda manual
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#1565C0', label='Feature original'),
                   Patch(facecolor='#E53935', label='Feature NUEVA')]
fig.legend(handles=legend_elements, loc='lower center', ncol=2, fontsize=12, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Top 15 Features — Clasificación (con nuevas features)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('./datasets/prepared/fig_feature_importance_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Resumen final y exportar resultados

In [ ]:
# ============================================================================
# RESUMEN FINAL
# ============================================================================

print('='*70)
print('RESUMEN — INGENIERÍA DE FEATURES AVANZADA')
print('='*70)

print(f'\nFeatures: {len(FEATURES_ORIGINAL)} originales + {len(FEATURES_NUEVAS)} nuevas = {len(features_disponibles)} total')

print(f'\n📊 CLASIFICACIÓN')
print(f'{"─"*60}')
df_c_final = pd.DataFrame(resultados_clasif_v2).T.sort_values('F1 (25 feat)', ascending=False)
for nombre in df_c_final.index:
    r = df_c_final.loc[nombre]
    emoji = '📈' if r['Mejora'] > 0.01 else '➡️'
    print(f'  {emoji} {nombre:20s}  {r["F1 (11 feat)"]:.4f} → {r["F1 (25 feat)"]:.4f} ({r["Mejora"]:+.4f})')

print(f'\n📈 REGRESIÓN')
print(f'{"─"*60}')
df_r_final = pd.DataFrame(resultados_regres_v2).T.sort_values('RMSE (25 feat)')
for nombre in df_r_final.index:
    r = df_r_final.loc[nombre]
    emoji = '📈' if r['Mejora RMSE'] > 0.1 else '➡️'
    print(f'  {emoji} {nombre:20s}  {r["RMSE (11 feat)"]:.3f} → {r["RMSE (25 feat)"]:.3f} ({r["Mejora RMSE"]:+.3f})')

# Guardar resultados
df_c_final.round(4).to_csv('./datasets/prepared/resultados_clasif_features_avanzadas.csv')
df_r_final.round(4).to_csv('./datasets/prepared/resultados_regres_features_avanzadas.csv')

# Guardar modelos
for nombre, modelo in modelos_clasif.items():
    with open(f'./datasets/prepared/modelo_v2_clasif_{nombre.lower().replace(" ", "_")}.pkl', 'wb') as f:
        pickle.dump(modelo, f)
for nombre, modelo in modelos_regres.items():
    with open(f'./datasets/prepared/modelo_v2_regres_{nombre.lower().replace(" ", "_")}.pkl', 'wb') as f:
        pickle.dump(modelo, f)

print(f'\n📁 Archivos guardados en ./datasets/prepared/')
print(f'\n✅ Notebook 5 completado')
print(f'\n🎯 PRÓXIMOS PASOS:')
print(f'   1. Evaluar modelos adicionales (SVM, Logistic Regression)')
print(f'   2. Análisis SHAP para interpretabilidad')
print(f'   3. Escribir Capítulos 4-7 del TFM con resultados de los 5 notebooks')

---

### Evolución completa del proyecto (para la memoria)

| Notebook | Fase | Resultado clave |
|----------|------|-----------------|
| **Nb1** | Recopilación de datos | 93k vueltas, 4 temporadas, FastF1 + Jolpica |
| **Nb2** | EDA y preparación | Desbalance 96.8/3.2%, división temporal |
| **Nb3** | Baseline (5 modelos × 2 tareas) | Mejor F1=0.711 (MLP), RMSE=8.39 (RF) |
| **Nb4** | Optimización hiperparámetros | Mejora marginal → cuello de botella = features |
| **Nb5** | Features avanzadas (11→25) | (resultados de este notebook) |

---

**Fin del Notebook 5.**

In [ ]:
# ============================================================================
# PERSISTIR DATASETS RECONSTRUIDOS CON LAS 25 FEATURES
# ============================================================================
# Esta celda guarda en disco los splits temporales con las 25 features avanzadas
# para poder reutilizarlos en el Notebook 9 (análisis SHAP) sin reejecutar
# toda la cadena de ingeniería de features.
 
import os
os.makedirs('./datasets/prepared', exist_ok=True)
 
# Columnas a guardar: metadatos + features + targets
# (se filtran las que realmente existan en cada split)
COLS_GUARDAR = (
    ['Season', 'Round', 'EventName', 'EventDate', 'Driver',  # metadatos
     'StopLapNumber', 'LapsUntilNextStop']                   # targets/auxiliares
    + features_disponibles                                    # las 25 features
    + ['target_parada']                                       # target de clasificación
)
 
def cols_existentes(df_, cols):
    """Devuelve solo las columnas que existen en el DataFrame, sin duplicados."""
    seen = set()
    out = []
    for c in cols:
        if c in df_.columns and c not in seen:
            out.append(c)
            seen.add(c)
    return out
 
# --- Persistir splits ---
ruta_train_clf = './datasets/prepared/train_clasificacion_avanzado.csv'
ruta_test_clf  = './datasets/prepared/test_clasificacion_avanzado.csv'
ruta_train_reg = './datasets/prepared/train_regresion_avanzado.csv'
ruta_test_reg  = './datasets/prepared/test_regresion_avanzado.csv'
 
train_c[cols_existentes(train_c, COLS_GUARDAR)].to_csv(ruta_train_clf, index=False)
test_c [cols_existentes(test_c,  COLS_GUARDAR)].to_csv(ruta_test_clf,  index=False)
train_r[cols_existentes(train_r, COLS_GUARDAR)].to_csv(ruta_train_reg, index=False)
test_r [cols_existentes(test_r,  COLS_GUARDAR)].to_csv(ruta_test_reg,  index=False)
 
# --- Persistir scalers también (útil si se quieren usar para MLP en NB9) ---
with open('./datasets/prepared/scaler_v2_clasif.pkl', 'wb') as f:
    pickle.dump(scaler_c, f)
with open('./datasets/prepared/scaler_v2_regres.pkl', 'wb') as f:
    pickle.dump(scaler_r, f)
 
#---Verificación--- 
print('✅ Datasets con 25 features persistidos:')
print(f'   {ruta_train_clf}')
print(f'      → {train_c.shape[0]:>6,} filas × {len(cols_existentes(train_c, COLS_GUARDAR))} cols')
print(f'   {ruta_test_clf}')
print(f'      → {test_c.shape[0]:>6,} filas × {len(cols_existentes(test_c, COLS_GUARDAR))} cols')
print(f'   {ruta_train_reg}')
print(f'      → {train_r.shape[0]:>6,} filas × {len(cols_existentes(train_r, COLS_GUARDAR))} cols')
print(f'   {ruta_test_reg}')
print(f'      → {test_r.shape[0]:>6,} filas × {len(cols_existentes(test_r, COLS_GUARDAR))} cols')
 
# --- Comprobación: ¿cuántas de las 25 features están en el dataset guardado? ---
n_feat_guardadas = sum(1 for f in features_disponibles if f in train_c.columns)
print(f'\n   Features del modelo en el dataset: {n_feat_guardadas}/{len(features_disponibles)}')
if n_feat_guardadas < len(features_disponibles):
    faltantes = [f for f in features_disponibles if f not in train_c.columns]
    print(f'   ⚠️ Faltantes: {faltantes}')
 